# Project 5: Post-Harvest Food Loss & Waste (FLW) & Scope 3 Carbon Diagnostic
## Data Pipeline: Pandas Data Hygiene, Scope 3 Carbon Accounting & Loss Mitigation Modeling

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# Task 1: Ingest Raw Shipment Log
df_raw = pd.read_csv('../data/food_loss_waste_raw.csv')
print('Raw Data Overview:')
print(f'Total Records: {len(df_raw)}')
df_raw.head()

## Task 2: Data Hygiene & Feature Engineering
We remove duplicate shipment records, standardize crop names and supply chain node strings, and derive physical loss %, financial loss (INR), and embodied Scope 3 carbon footprint (Ton CO2e).

In [ ]:
# Drop duplicates
df_clean = df_raw.drop_duplicates().copy()

# Standardize text formatting
df_clean['crop_name'] = df_clean['crop_name'].astype(str).str.strip().str.title()
df_clean['supply_chain_node'] = df_clean['supply_chain_node'].astype(str).str.strip().str.title()

# Impute missing ambient temperatures with node-level median
df_clean['ambient_temp_c'] = df_clean.groupby('supply_chain_node')['ambient_temp_c'].transform(lambda x: x.fillna(x.median()))

# Feature Engineering
df_clean['physical_loss_pct'] = (df_clean['spoilage_tonnage'] / df_clean['initial_tonnage'] * 100).round(2)
df_clean['financial_loss_inr'] = (df_clean['spoilage_tonnage'] * df_clean['export_price_per_ton_inr']).round(2)
df_clean['embodied_ghg_loss_kg_co2e'] = (df_clean['spoilage_tonnage'] * df_clean['ghg_factor_kg_co2_per_ton']).round(2)
df_clean['embodied_ghg_loss_ton_co2e'] = (df_clean['embodied_ghg_loss_kg_co2e'] / 1000.0).round(2)

print(f'Clean Records: {len(df_clean)}')
print(f'Total Spoilage Tonnage: {df_clean["spoilage_tonnage"].sum():,.2f} Tons')
print(f'Total Financial Loss: INR {df_clean["financial_loss_inr"].sum():,.2f}')
print(f'Total Scope 3 Carbon Loss: {df_clean["embodied_ghg_loss_ton_co2e"].sum():,.2f} Ton CO2e')

## Task 3: Supply Chain Node Aggregation & Export
We aggregate financial losses and embodied carbon emissions by supply chain node and export summary statistics.

In [ ]:
node_summary = df_clean.groupby('supply_chain_node').agg(
    total_shipments=('shipment_id', 'count'),
    total_initial_tonnage=('initial_tonnage', 'sum'),
    total_spoilage_tonnage=('spoilage_tonnage', 'sum'),
    avg_loss_pct=('physical_loss_pct', 'mean'),
    total_financial_loss_inr=('financial_loss_inr', 'sum'),
    total_ghg_loss_ton_co2e=('embodied_ghg_loss_ton_co2e', 'sum')
).reset_index()

print('=== SUPPLY CHAIN NODE SUMMARY ===')
print(node_summary.to_string(index=False))